In [1]:
import pandas as pd
import os

In [2]:
os.chdir("C:\\Users\\Aulbek\\Рабочий стол\\DjangoProjects\\archive")
# Получаем текущую директорию
folder_path = os.getcwd()
file_names = ["channels.csv","deliveries.csv","drivers.csv","hubs.csv","orders.csv","payments.csv","stores.csv"]

In [3]:
dfs = {file: pd.read_csv(os.path.join(folder_path, file)) for file in file_names}
# 🛠 Проверка и обработка данных
cleaned_dfs = {}  # Словарь для чистых данных

In [4]:
# Проверяем первые строки каждого DataFrame
for file, df in dfs.items():
     print(f"\n🔍 Анализ {file}:")
     # 📊 Основная информация
     print(df.info())
     print("Пропущенные значения:\n", df.isnull().sum())
     # 👥 Проверяем дубликаты
     num_duplicates = df.duplicated().sum()
     print(f"👥 Найдено дубликатов: {num_duplicates}")
     # Удаляем дубликаты
     df = df.drop_duplicates()
    
    # 🛑 Удаляем столбцы, где >30% данных отсутствует
threshold = 0.3  # 30%
for file, df in dfs.items():
    missing_ratio = df.isnull().mean()  # Доля пропущенных значений в каждом столбце
    cleaned_df = df.loc[:, missing_ratio < threshold]  # Удаляем столбцы с более чем 30% NaN

    if cleaned_df.shape[1] > 0:  # Проверяем, остались ли столбцы
        cleaned_dfs[file] = cleaned_df
        print(f"✅ {file} сохранён, столбцы: {cleaned_df.shape[1]}")
    else:
        print(f"❌ {file} УДАЛЁН, так как все столбцы содержали >30% пропущенных данных")
    


🔍 Анализ channels.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    40 non-null     int64 
 1   channel_name  40 non-null     object
 2   channel_type  40 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.1+ KB
None
Пропущенные значения:
 channel_id      0
channel_name    0
channel_type    0
dtype: int64
👥 Найдено дубликатов: 0

🔍 Анализ deliveries.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378843 entries, 0 to 378842
Data columns (total 5 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   delivery_id               378843 non-null  int64  
 1   delivery_order_id         378843 non-null  int64  
 2   driver_id                 362957 non-null  float64
 3   delivery_distance_meters  378770 non-null  float64
 4   delivery_status   

In [5]:
# 🔍 Проверяем названия колонок
print("📌 Столбцы в orders.csv:", cleaned_dfs["orders.csv"].columns)
print("📌 Столбцы в payments.csv:", cleaned_dfs["payments.csv"].columns)

# 🔄 Переименовываем payment_order_id → order_id в payments.csv
cleaned_dfs["payments.csv"].rename(columns={"payment_order_id": "order_id"}, inplace=True)

# 🛑 Создаём копию перед изменением
cleaned_dfs["orders.csv"] = cleaned_dfs["orders.csv"].copy()
cleaned_dfs["payments.csv"] = cleaned_dfs["payments.csv"].copy()

# 🔄 Приводим order_id к строковому типу (чтобы избежать проблем при объединении)
cleaned_dfs["orders.csv"]["order_id"] = cleaned_dfs["orders.csv"]["order_id"].astype(str)
cleaned_dfs["payments.csv"]["order_id"] = cleaned_dfs["payments.csv"]["order_id"].astype(str)

# 🔗 Объединяем по order_id
merged_df = pd.merge(cleaned_dfs["orders.csv"], cleaned_dfs["payments.csv"], on="order_id", how="inner", suffixes=('_orders', '_payments'))

print(f"\n🔗 Объединены orders.csv и payments.csv, итог: {merged_df.shape}")


📌 Столбцы в orders.csv: Index(['order_id', 'store_id', 'channel_id', 'payment_order_id',
       'delivery_order_id', 'order_status', 'order_amount',
       'order_delivery_fee', 'order_delivery_cost', 'order_created_hour',
       'order_created_minute', 'order_created_day', 'order_created_month',
       'order_created_year', 'order_moment_created', 'order_moment_accepted',
       'order_moment_ready', 'order_moment_collected',
       'order_moment_in_expedition', 'order_moment_delivering',
       'order_moment_finished', 'order_metric_collected_time',
       'order_metric_paused_time', 'order_metric_production_time',
       'order_metric_walking_time', 'order_metric_expediton_speed_time',
       'order_metric_transit_time', 'order_metric_cycle_time'],
      dtype='object')
📌 Столбцы в payments.csv: Index(['payment_id', 'payment_order_id', 'payment_amount', 'payment_fee',
       'payment_method', 'payment_status'],
      dtype='object')

🔗 Объединены orders.csv и payments.csv, итог: (40

In [6]:
merged_df.head()  # Покажет первые 5 строк


,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,...,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time,payment_id,payment_amount,payment_fee,payment_method,payment_status
0,68410055,2181,35,68410055,68410055,FINISHED,394.80,0.01,6.00,2,...,2391.25,7.17,11.72,21.75,2424.72,4427917,118.44,0.00,VOUCHER,PAID
1,68410055,2181,35,68410055,68410055,FINISHED,394.80,0.01,6.00,2,...,2391.25,7.17,11.72,21.75,2424.72,4427918,394.81,7.90,ONLINE,PAID
2,68412721,631,5,68412721,68412721,FINISHED,195.05,11.90,10.93,14,...,26.07,0.83,11.05,83.30,120.42,4427941,206.95,5.59,ONLINE,PAID
3,68413340,631,5,68413340,68413340,FINISHED,46.90,11.90,11.36,14,...,14.62,9.57,12.67,49.78,77.05,4427948,58.80,1.59,ONLINE,PAID
4,68414018,3265,5,68414018,68414018,FINISHED,45.80,0.00,10.28,14,...,27.48,10.25,13.28,11.05,51.82,4427955,45.80,0.92,ONLINE,PAID
